In [2]:
%pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load dataset
df = pd.read_csv("Dataset_14-day_AA_depression_symptoms_mood_and_PHQ-9.csv")
df.columns = df.columns.str.strip()

In [5]:
phq_columns = [f"phq{i}" for i in range(1, 10)]

# EMA daily questions (legitimate features)
ema_columns = [
    "q1", "q2", "q3", "q4", "q5", "q6", "q7", "q8", "q9",
    "q10", "q11", "q12", "q13", "q14", "q16", "q46", "q47"
]


In [6]:
# Basic demographics
demo_columns = ["user_id", "age", "sex"]

# Time-related columns
time_columns = ["time", "phq.day", "period.name"]

# Target/scores
score_columns = ["happiness.score"]

# All columns to keep
keep_columns = demo_columns + time_columns + score_columns + phq_columns + ema_columns

# Filter to existing columns
keep_columns = [col for col in keep_columns if col in df.columns]
df = df[keep_columns]

In [7]:
# Convert PHQ columns to numeric
for col in phq_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate PHQ total score (0-27)
df["phq_total"] = df[phq_columns].sum(axis=1)

In [8]:
def get_depression_level(score):
    if pd.isna(score):
        return np.nan
    elif score <= 4:
        return 0  # Minimal
    elif score <= 9:
        return 1  # Mild
    elif score <= 14:
        return 2  # Moderate
    else:
        return 3  # Severe

df["depression_level"] = df["phq_total"].apply(get_depression_level)

In [ ]:
# Drop rows with no target
df = df.dropna(subset=["depression_level"])

In [ ]:
print("\n📊 DEPRESSION DISTRIBUTION:")
print(df["depression_level"].value_counts().sort_index())
print(f"\nPercentages:")
print(df["depression_level"].value_counts(normalize=True).sort_index().mul(100).round(1))

In [ ]:
# Convert EMA questions to numeric
for col in ema_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing EMA values with median
for col in ema_columns:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Create "answered" flags (captures missingness pattern)
for col in ema_columns:
    if col in df.columns:
        df[f"{col}_answered"] = df[col].notna().astype(int)

print(f"\n Processed {len(ema_columns)} EMA questions")


In [ ]:
df["age"] = pd.to_numeric(df["age"], errors="coerce")
df["age"] = df["age"].fillna(df["age"].median())

# Sex (encode)
df["sex"] = df["sex"].fillna(df["sex"].mode()[0] if len(df["sex"].mode()) > 0 else 0)
df["sex"] = df["sex"].map({"male": 0, "female": 1, "Male": 0, "Female": 1})
df["sex"] = df["sex"].fillna(0).astype(int)

# Happiness score
df["happiness.score"] = pd.to_numeric(df["happiness.score"], errors="coerce")
df["happiness.score"] = df["happiness.score"].fillna(df["happiness.score"].median())

print(f"\n  Processed demographics")
print(f"  - Age range: {df['age'].min():.0f} to {df['age'].max():.0f}")
print(f"  - Sex distribution: {df['sex'].value_counts().to_dict()}")


In [ ]:
if "time" in df.columns:
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    
    # Extract time features
    df["hour"] = df["time"].dt.hour.fillna(12).astype(int)
    df["day_of_week"] = df["time"].dt.dayofweek.fillna(0).astype(int)
    df["month"] = df["time"].dt.month.fillna(1).astype(int)
    
    # Sort by user and time (important for time series)
    df = df.sort_values(["user_id", "time"])
    
    # Drop original time column
    df = df.drop(columns=["time"])


In [ ]:
# Process phq.day
df["phq.day"] = pd.to_numeric(df["phq.day"], errors="coerce")
df["phq.day"] = df["phq.day"].fillna(df["phq.day"].median())

# Encode period.name
if "period.name" in df.columns:
    df["period.name"] = df["period.name"].fillna("unknown")
    period_dummies = pd.get_dummies(df["period.name"], prefix="period", dtype=int)
    df = pd.concat([df, period_dummies], axis=1)
    df = df.drop(columns=["period.name"])

print(f"\n✅ Processed time features")

In [ ]:
df = df.sort_values(["user_id", "phq.day"])

# Previous happiness (yesterday's mood)
df["prev_happiness"] = df.groupby("user_id")["happiness.score"].shift(1)

# Rolling average happiness (last 3 days)
df["rolling_happiness"] = df.groupby("user_id")["happiness.score"].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# Happiness trend (change from previous)
df["happiness_trend"] = df.groupby("user_id")["happiness.score"].diff()

# Time difference between entries
if "phq.day" in df.columns:
    df["days_since_last"] = df.groupby("user_id")["phq.day"].diff()

In [ ]:
# Fill NaN values for time-series features
df["prev_happiness"] = df["prev_happiness"].fillna(df["happiness.score"])
df["rolling_happiness"] = df["rolling_happiness"].fillna(df["happiness.score"])
df["happiness_trend"] = df["happiness_trend"].fillna(0)
if "days_since_last" in df.columns:
    df["days_since_last"] = df["days_since_last"].fillna(0)

print(f"\n✅ Created time-series features")

In [ ]:
columns_to_remove = [
    # Identifiers
    "user_id",
    
    # Target leakage (PHQ questions - these ARE the test!)
    *phq_columns,
    
    # Target itself
    "phq_total",
    "depression_level",
    
    # Weak features (optional - can keep if you want)
    "month",
    "day_of_week",
]

# Remove answered flags if too many (optional)
answered_flags = [col for col in df.columns if col.endswith("_answered")]
# columns_to_remove.extend(answered_flags)  # Uncomment if you want to remove them

# Remove only columns that exist
columns_to_remove = [col for col in columns_to_remove if col in df.columns]


In [ ]:
X = df.drop(columns=columns_to_remove)
y = df["depression_level"]

print("="*60)
print(" DATA LEAKAGE FIXED")
print("="*60)
print(f"Features kept: {len(X.columns)}")
print(f"\nFEATURES USED FOR TRAINING:")
for i, col in enumerate(X.columns, 1):
    print(f"  {i:2d}. {col}")

# %% [markdown]
# # 10. FINAL DATA QUALITY CHECKS

# %%
print("\n" + "="*60)
print("DATA QUALITY REPORT")
print("="*60)

print(f"\n Dataset shape: {X.shape}")
print(f" Target classes: {y.nunique()} (0=Minimal, 1=Mild, 2=Moderate, 3=Severe)")
print(f"\n Missing values in X: {X.isnull().sum().sum()}")
print(f" Missing values in y: {y.isnull().sum()}")

# Check for any remaining PHQ columns
remaining_phq = [col for col in X.columns if col.startswith("phq")]
if remaining_phq:
    print(f"\n WARNING: Still have PHQ columns in X: {remaining_phq}")
else:
    print(f"\n No PHQ columns in features (good!)")

# Check class distribution
print(f"\n Class distribution:")
class_dist = y.value_counts().sort_index()
for i, count in class_dist.items():
    labels = ["Minimal", "Mild", "Moderate", "Severe"]
    print(f"  {labels[i]}: {count} ({count/len(y)*100:.1f}%)")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n Train set: {X_train.shape}")
print(f" Test set: {X_test.shape}")

# %% [markdown]
# # 12. TRAIN MODEL WITH CLASS WEIGHTS

# %%
# Handle class imbalance
classes = np.unique(y)
class_weights = compute_class_weight(
    'balanced', 
    classes=classes, 
    y=y
)
weight_dict = dict(zip(classes, class_weights))

print("\n Class weights applied:")
for level, weight in weight_dict.items():
    labels = ["Minimal", "Mild", "Moderate", "Severe"]
    print(f"  {labels[level]}: {weight:.3f}")